# RAG Knowledge Base Builder (OpenAI/Euron Version)

This notebook is responsible for compiling and updating the RAG knowledge base for your portfolio's chatbot using OpenAI/Euron API. It processes project details, experiences, and blog data from YAML configuration files, parses local resume PDFs, generates dense vector embeddings using OpenAI's `text-embedding-3-small` model, and saves the final vector database to disk.

In [ ]:
# 1. Install Required Python Packages
!pip install -q pypdf langchain langchain-community python-dotenv pyyaml openai

In [ ]:
# 2. Initialize Directories and Environment Configurations
import os
import yaml
import time
import json
import glob
from dotenv import load_dotenv

# Load environment variables from the root .env file
load_dotenv(dotenv_path=".env")

# Target Directory Paths
resumes_in_dir = "data/resumes"
knowledge_dir = "data/knowledge"
vector_out_dir = "public/data/knowledge"

# Create directories automatically if they do not exist
os.makedirs(resumes_in_dir, exist_ok=True)
os.makedirs(knowledge_dir, exist_ok=True)
os.makedirs(vector_out_dir, exist_ok=True)

# Load API parameters (default to Euron.one API configurations)
api_key = os.getenv("VITE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("VITE_OPENAI_BASE_URL") or os.getenv("OPENAI_BASE_URL") or "https://api.openai.com/v1"

if not api_key:
    api_key = input("⚠️ OpenAI API Key not found in environment. Please enter your API Key: ").strip()
    if api_key:
        save_env = input("Would you like to save this key to your .env file? (y/n): ").strip().lower()
        if save_env == 'y':
            with open(".env", "a") as env_file:
                env_file.write(f"\nVITE_OPENAI_API_KEY={api_key}\n")
            print("✅ API Key successfully written to .env!")
else:
    print(f"✅ API Key loaded: {api_key[:10]}...")
    print(f"✅ Base URL loaded: {base_url}")

In [ ]:
# 3. Extract and Consolidate Text from YAML Files
yaml_sources = [
    ("src/data/portfolio.yaml", "Portfolio Profile"),
    ("src/data/projects.yaml", "Projects and Tech Stack"),
    ("src/data/blog.yaml", "Blogs and Articles")
]

consolidated_blocks = []

for filepath, name in yaml_sources:
    if os.path.exists(filepath):
        print(f"Parsing {name} from {filepath}...")
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = yaml.safe_load(f)
                # Convert YAML content to structured readable text
                text_block = f"=== CONTENT SOURCE: {name.upper()} ===\n"
                text_block += yaml.dump(content, default_flow_style=False, allow_unicode=True)
                consolidated_blocks.append(text_block)
        except Exception as e:
            print(f"❌ Error parsing {filepath}: {e}")
    else:
        print(f"⚠️ Warning: {filepath} not found. Skipping.")

knowledge_base_path = os.path.join(knowledge_dir, "knowledge_base.txt")
with open(knowledge_base_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(consolidated_blocks))

print(f"\n✅ Created consolidated knowledge text file at: {knowledge_base_path} ({len(consolidated_blocks)} sections parsed)")

In [ ]:
# 4. Load PDFs from data/resumes/
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = glob.glob(os.path.join(resumes_in_dir, "*.pdf"))
resume_documents = []

if not pdf_paths:
    print("\n" + "!"*90)
    print("⚠️ No resume PDFs were found in the data/resumes folder.")
    print("Please add one or more resume PDF files before building the RAG knowledge base.")
    print("!"*90 + "\n")
else:
    print(f"Found {len(pdf_paths)} PDF(s) in {resumes_in_dir}. Parsing...")
    for path in pdf_paths:
        filename = os.path.basename(path)
        try:
            print(f"Loading {filename} via PyPDFLoader...")
            loader = PyPDFLoader(path)
            pages = loader.load()
            print(f"└─ Loaded {len(pages)} pages")
            resume_documents.extend(pages)
        except Exception as e:
            print(f"❌ Error loading {filename}: {e}")

In [ ]:
# 5. Create Text Chunks from Sources
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks_list = []

# Split YAML knowledge base
if os.path.exists(knowledge_base_path):
    with open(knowledge_base_path, "r", encoding="utf-8") as f:
        kb_content = f.read()
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    yaml_chunks = text_splitter.create_documents(
        [kb_content],
        metadatas=[{"source": "yaml_knowledge_base"}]
    )
    chunks_list.extend(yaml_chunks)
    print(f"Split knowledge_base.txt into {len(yaml_chunks)} chunks.")

# Split PDF resumes
if resume_documents:
    pdf_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100
    )
    pdf_chunks = pdf_splitter.split_documents(resume_documents)
    chunks_list.extend(pdf_chunks)
    print(f"Split resumes PDFs into {len(pdf_chunks)} chunks.")

print(f"\n✅ Total chunks prepared for embedding: {len(chunks_list)}")

In [ ]:
# 6. Generate and Save Vector Embeddings using OpenAI/Euron API
from openai import OpenAI

if not api_key:
    raise ValueError("❌ OpenAI/Euron API Key is required to run the embedding pipeline.")

client = OpenAI(api_key=api_key, base_url=base_url)

print(f"Embedding {len(chunks_list)} text chunks using Euron.one API (text-embedding-3-small)...")
vector_database = []

for idx, chunk in enumerate(chunks_list):
    text = chunk.page_content.strip()
    if not text:
        continue
        
    source = chunk.metadata.get("source", "unknown")
    page = chunk.metadata.get("page", 0)
    
    # Standard short delay
    time.sleep(0.02)
        
    try:
        res = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        vector_database.append({
            "id": f"chunk_{idx}",
            "text": text,
            "embedding": res.data[0].embedding,
            "metadata": {
                "source": source,
                "page": page
            }
        })
        if (idx + 1) % 20 == 0:
            print(f"Embedded {idx + 1} / {len(chunks_list)} chunks...")
    except Exception as e:
        print(f"❌ Error embedding chunk {idx}: {e}")

vector_db_path = os.path.join(vector_out_dir, "vector_db.json")
with open(vector_db_path, "w", encoding="utf-8") as f:
    json.dump(vector_database, f, indent=2)

print("\n" + "="*90)
print(f"🎉 SUCCESS! Created vector database with {len(vector_database)} entries.")
print(f"Saved to: {vector_db_path}")
print("="*90)
